# FINAL GATE：資料湖 → 獨立回測 Agent 最終整合驗收（唯讀，不需網路）

讀取固定版本：`fubon_5m` seq 68、quality 1.0.0 遮罩、B3a／B4a／B5a 正式輸出，逐一核對 SHA-256。
以固定、不調參的探針跑完整條路徑：5m→30m 重採樣、時間可用性與防前視、PIT 選股、公司行動持倉會計、漲跌停／UNKNOWN／停牌／無法成交、交易 ledger（含 INVALIDATED）、成本與滑價，並重跑一次確認結果完全一致。
最後機械式產生 **GO／SCOPED GO／NO-GO**，以及 `FINAL_READINESS_REPORT.md`、`SCOPE_MATRIX.csv`、`BLOCKER_REGISTER.csv`、`run_manifest.json`。

不修改 lake 任何檔案；結果寫在 `audits/final_gate/`。**依序執行全部格子即可**，約 20–40 分鐘。

In [ ]:
# 1. 參數
REPO_OWNER, REPO_NAME, BRANCH = "hh4832", "tw-market-data-lake", "main"
DATA_ROOT = "/content/drive/MyDrive/00Quant_Research/data"
PROJECT_DIR = f"/content/{REPO_NAME}"
OUT_DIR = "/content/tw_lake_final_gate"
ARCHIVE = f"{DATA_ROOT}/_tw-market-data-lake/audits/final_gate"
N_DATES, K, SEED = 30, 10, 20261031      # 固定，不得依結果調整


In [ ]:
# 2. 掛載 Google Drive
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
# 3. clone / pull + 依賴
import subprocess, os
if os.path.exists(f"{PROJECT_DIR}/.git"):
    subprocess.run(["git", "-C", PROJECT_DIR, "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", PROJECT_DIR, "checkout", "-B", BRANCH, f"origin/{BRANCH}"], check=True)
else:
    subprocess.run(["git", "clone", "--branch", BRANCH, f"https://github.com/{REPO_OWNER}/{REPO_NAME}.git", PROJECT_DIR], check=True)
print("HEAD:", subprocess.check_output(["git", "-C", PROJECT_DIR, "rev-parse", "HEAD"], text=True).strip())
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml>=6", "tqdm>=4.66"], check=True)
def stream(args):
    env = dict(os.environ, PYTHONPATH=f"{PROJECT_DIR}/src", PYTHONUNBUFFERED="1")
    p = subprocess.Popen(args, cwd=PROJECT_DIR, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="")
    return p.wait()

def ensure_drive():
    """Mount Drive safely. If an earlier cell unmounted Drive and then wrote under /content/drive,
    those files are only on this runtime's disk: move them aside (never delete) and copy any audit
    folders that are missing on the real Drive."""
    import os, shutil, time
    from google.colab import drive
    if os.path.ismount("/content/drive"):
        return
    stray = None
    if os.path.isdir("/content/drive") and os.listdir("/content/drive"):
        stray = f"/content/drive_local_{int(time.time())}"
        shutil.move("/content/drive", stray)
        print("moved local (unsynced) files to", stray)
    drive.mount("/content/drive")
    if stray:
        src_root = f"{stray}/MyDrive/00Quant_Research/data/_tw-market-data-lake/audits"
        dst_root = f"{DATA_ROOT}/_tw-market-data-lake/audits"
        for dirpath, dirnames, filenames in os.walk(src_root):
            rel = os.path.relpath(dirpath, src_root)
            for d in list(dirnames):
                target = os.path.join(dst_root, rel, d)
                if not os.path.exists(target):
                    shutil.copytree(os.path.join(dirpath, d), target)
                    print("copied missing", target)
                    dirnames.remove(d)

In [ ]:
# 4. 執行
env_pp = f"{PROJECT_DIR}:{PROJECT_DIR}/src"
import os
os.environ["PYTHONPATH"] = env_pp
def stream2(args):
    env = dict(os.environ, PYTHONPATH=env_pp, PYTHONUNBUFFERED="1")
    p = subprocess.Popen(args, cwd=PROJECT_DIR, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="")
    return p.wait()
print("exit", stream2([sys.executable, "-m", "acceptance.final_gate.run", "--data-root", DATA_ROOT,
                       "--out-root", OUT_DIR, "--dates", str(N_DATES), "--k", str(K), "--seed", str(SEED)]))


In [ ]:
# 5. 結果，並複製到 Drive（把這格輸出貼給 Claude）
import glob, json, os, shutil
import pandas as pd
ensure_drive()
pd.set_option("display.width", 250); pd.set_option("display.max_colwidth", 120); pd.set_option("display.max_rows", 200)
d = sorted(glob.glob(f"{OUT_DIR}/final_gate_*"))[-1]
print(open(f"{d}/FINAL_READINESS_REPORT.md", encoding="utf-8").read())
sc = pd.read_csv(f"{d}/SCOPE_MATRIX.csv")
print("\n== SCOPE_MATRIX (retention by liquidity / year)"); print(sc[["scope_id", "status", "retention", "retention_by_liquidity", "retention_by_year", "top_exclusion_reasons"]].to_string(index=False))
ch = json.load(open(f"{d}/checks.json", encoding="utf-8"))
print("\n== failed checks:", {k: v["detail"] for k, v in ch.items() if not v["passed"]})
led = pd.read_parquet(f"{d}/probe_ledger.parquet")
print("\n== multi-day trades with corporate actions"); print(led[(led.strategy == "MULTI_DAY") & (led.n_events > 0)][["symbol", "signal_date", "status", "n_events", "cash_received", "shares_exit", "acc_return_open", "adj_return_open"]].head(20).to_string(index=False))
m = json.load(open(f"{d}/run_manifest.json", encoding="utf-8"))
print("\n== pins"); print(json.dumps({k: {kk: vv for kk, vv in v.items() if kk != "sha256"} if isinstance(v, dict) else v for k, v in m["pins"].items()}, ensure_ascii=False, indent=1, default=str)[:4000])
print("\nVERDICT:", m["verdict"], "| ledger sha", m["ledger_sha256"])
os.makedirs(ARCHIVE, exist_ok=True)
dst = f"{ARCHIVE}/{os.path.basename(d)}"
if not os.path.exists(dst):
    shutil.copytree(d, dst)
print("copied to", dst)
